merge the signal and EPD histograms for all batches, and combine the average Nch data

adjust batches_dir and out_file as necessary

In [ ]:
import ROOT
import numpy as np
import math
import matplotlib.pyplot as plt
import os

In [2]:
ROOT.gDirectory.Clear()

In [3]:
batches_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/per_batch"


In [4]:
analysis_bins = [ [0,25], [25,36], [36,48], [48,60], [60,71], [71,78], [78,91], [91,97], [97,1000] ]

In [5]:
# opening relevant histograms and parameters from filepath

def openHistograms(filepath):
    '''
    takes a filepath and returns (per mult bin, for both wta and std)
    - signal
    - background
    - total Nch
    - number of jets
    '''

    file = ROOT.TFile.Open(filepath, "READ")

    wta_sig_hists = {}
    std_sig_hists = {}
    
    #wta_bkg_hists = {}
    #std_bkg_hists = {}

    wta_total_Nch_dict = {}
    std_total_Nch_dict = {}

    wta_num_jets_dict = {}
    std_num_jets_dict = {}

    wta_epd_hists = {}
    std_epd_hists = {}

    for mult_bin in analysis_bins:
        bin_key = f"{mult_bin[0]} <  Nch < {mult_bin[1]}"

        # open histograms
        hSig_wta = file.Get(f"hSig_WTA_{mult_bin[0]}_{mult_bin[1]}")
        hSig_std = file.Get(f"hSig_STD_{mult_bin[0]}_{mult_bin[1]}")
        
        #hBkg_wta = file.Get(f"hBkg_WTA_{mult_bin[0]}_{mult_bin[1]}")
        #hBkg_std = file.Get(f"hBkg_STD_{mult_bin[0]}_{mult_bin[1]}")

        hEPD_wta = file.Get(f"hEPD_WTA_{mult_bin[0]}_{mult_bin[1]}")
        hEPD_std = file.Get(f"hEPD_STD_{mult_bin[0]}_{mult_bin[1]}")

        # detach them
        hSig_wta.SetDirectory(0)
        hSig_std.SetDirectory(0)
        
        #hBkg_wta.SetDirectory(0)
        #hBkg_std.SetDirectory(0)

        hEPD_wta.SetDirectory(0)
        hEPD_std.SetDirectory(0)

        # add to dictionaries
        wta_sig_hists[bin_key] = hSig_wta
        std_sig_hists[bin_key] = hSig_std

        #wta_bkg_hists[bin_key] = hBkg_wta
        #std_bkg_hists[bin_key] = hBkg_std

        wta_epd_hists[bin_key] = hEPD_wta
        std_epd_hists[bin_key] = hEPD_std


        # read parameters
        total_Nch_wta = file.Get(f"total_Nch_WTA_{mult_bin[0]}_{mult_bin[1]}").GetVal()
        total_Nch_std = file.Get(f"total_Nch_STD_{mult_bin[0]}_{mult_bin[1]}").GetVal()

        num_jets_wta = file.Get(f"num_jets_WTA_{mult_bin[0]}_{mult_bin[1]}").GetVal()
        num_jets_std = file.Get(f"num_jets_STD_{mult_bin[0]}_{mult_bin[1]}").GetVal()

        wta_num_jets_dict[bin_key] = num_jets_wta
        std_num_jets_dict[bin_key] = num_jets_std

        wta_total_Nch_dict[bin_key] = total_Nch_wta
        std_total_Nch_dict[bin_key] = total_Nch_std
        
    signals = {"wta": wta_sig_hists, "std": std_sig_hists}
    #backgrounds = {'wta': wta_bkg_hists, 'std': std_bkg_hists}
    EPDs = {'wta': wta_epd_hists, 'std': std_epd_hists}
    total_Nch = {'wta': wta_total_Nch_dict, 'std': std_total_Nch_dict}
    num_jets = {'wta': wta_num_jets_dict, 'std': std_num_jets_dict}

    return signals, EPDs, total_Nch, num_jets



In [6]:
# initialising merged histograms

merged_wta_sig_hists = {}
merged_std_sig_hists = {}

#merged_wta_bkg_hists = {}
#merged_std_bkg_hists = {}

merged_wta_epd_hists = {}
merged_std_epd_hists = {}

merged_wta_total_Nch = {}
merged_std_total_Nch = {}

merged_wta_num_jets = {}
merged_std_num_jets = {}

# eta and phi bins
eta_bins, eta_min, eta_max = 41, -6.15, 6.15    # from xiao's code. prl paper uses -3 to 3
phi_bins = 33
phi_min = -(math.pi/2.0) + (math.pi/32.0)
phi_max = (3*math.pi/2.0) + (math.pi/32.0)
phi_bin_width = (phi_max - phi_min)/phi_bins

for mult_bin in analysis_bins:
    bin_key = f"{mult_bin[0]} <  Nch < {mult_bin[1]}"

    safe_name = f"{mult_bin[0]}_{mult_bin[1]}"
   
    # 1. INITIALISE HISTOGRAMS (TH2D)
    merged_wta_sig_hists[bin_key] = ROOT.TH2D(f"WTA_sig_{safe_name}", f"WTA Signal ({bin_key});#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
    merged_std_sig_hists[bin_key] = ROOT.TH2D(f"STD_sig_{safe_name}", f"Standard Signal ({bin_key});#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
    
    #merged_wta_bkg_hists[bin_key] = ROOT.TH2D(f"WTA_bkg_{safe_name}", f"WTA Background ({bin_key});#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
    #merged_std_bkg_hists[bin_key] = ROOT.TH2D(f"STD_bkg_{safe_name}", f"Standard Background ({bin_key});#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)

    merged_wta_epd_hists[bin_key] = ROOT.TH2D(f"WTA_epd_{safe_name}", f"WTA EPD ({bin_key});#eta*;#phi*", 150, 0, 10, 120, -4, 4)
    merged_std_epd_hists[bin_key] = ROOT.TH2D(f"STD_epd_{safe_name}", f"Standard EPD ({bin_key});#eta*;#phi*", 150, 0, 10, 120, -4, 4)

    # initialise param values
    merged_wta_total_Nch[bin_key] = 0
    merged_std_total_Nch[bin_key] = 0

    merged_wta_num_jets[bin_key] = 0
    merged_std_num_jets[bin_key] = 0

In [8]:
# merging data


for batch_num in range(18):

    if batch_num == 4 or batch_num == 5 or batch_num == 6 or batch_num == 9:
        continue
    
    filename = f"Analysis_Output_batch{batch_num}.root"

    batch_file_path = os.path.join(batches_dir, filename)

    batch_sigs, batch_epds, batch_total_Nch, batch_num_jets = openHistograms(batch_file_path)

    for mult_bin in analysis_bins:
        bin_key = f"{mult_bin[0]} <  Nch < {mult_bin[1]}"
        
        # add up signal, background, and EPDs
        merged_wta_sig_hists[bin_key].Add(batch_sigs['wta'][bin_key])
        merged_std_sig_hists[bin_key].Add(batch_sigs['std'][bin_key])

        #merged_wta_bkg_hists[bin_key].Add(batch_bkgs['wta'][bin_key])
        #merged_std_bkg_hists[bin_key].Add(batch_bkgs['std'][bin_key])

        merged_wta_epd_hists[bin_key].Add(batch_epds['wta'][bin_key])
        merged_std_epd_hists[bin_key].Add(batch_epds['std'][bin_key])

        # add up total Nch and num jets
        merged_wta_total_Nch[bin_key] += batch_total_Nch['wta'][bin_key]
        merged_std_total_Nch[bin_key] += batch_total_Nch['std'][bin_key]

        merged_wta_num_jets[bin_key] += batch_num_jets['wta'][bin_key]
        merged_std_num_jets[bin_key] += batch_num_jets['std'][bin_key]




In [9]:
# true avg Nch values

merged_wta_avg_Nch = {}
merged_std_avg_Nch = {}


for mult_bin in analysis_bins:
    bin_key = f"{mult_bin[0]} <  Nch < {mult_bin[1]}"

    merged_wta_avg_Nch[bin_key] = merged_wta_total_Nch[bin_key] / merged_wta_num_jets[bin_key]
    merged_std_avg_Nch[bin_key] = merged_std_total_Nch[bin_key] / merged_std_num_jets[bin_key]
    

In [10]:
# saving merged histograms. 
# EDIT FILE PATH IF NECESSARY

out_file = ROOT.TFile("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/per_batch/merged_signals.root", "RECREATE")

for mult_bin in analysis_bins:
    bin_key = f"{mult_bin[0]} <  Nch < {mult_bin[1]}"

    # write histograms
    merged_wta_sig_hists[bin_key].Write()
    merged_std_sig_hists[bin_key].Write()

    #merged_wta_bkg_hists[bin_key].Write()
    #merged_std_bkg_hists[bin_key].Write()

    merged_wta_epd_hists[bin_key].Write()
    merged_std_epd_hists[bin_key].Write()

    # write TParams
    param_avg_Nch_wta = ROOT.TParameter('double')(f"avg_Nch_WTA_{mult_bin[0]}_{mult_bin[1]}", merged_wta_avg_Nch[bin_key])
    param_avg_Nch_std = ROOT.TParameter('double')(f"avg_Nch_STD_{mult_bin[0]}_{mult_bin[1]}", merged_std_avg_Nch[bin_key])
    param_avg_Nch_wta.Write()
    param_avg_Nch_std.Write()

    param_num_jets_wta = ROOT.TParameter('double')(f"num_jets_WTA_{mult_bin[0]}_{mult_bin[1]}", merged_wta_num_jets[bin_key])
    param_num_jets_std = ROOT.TParameter('double')(f"num_jets_STD_{mult_bin[0]}_{mult_bin[1]}", merged_std_num_jets[bin_key])
    param_num_jets_wta.Write()
    param_num_jets_std.Write()

out_file.Close()
